# EDA-Genius: Phase 1 Exploratory Data Analysis & Semiconductor Analytics

This notebook analyzes the relationship between **RTL structural complexity**, **synthesized netlist characteristics**, **physical design configurations**, and **EDA outcomes** (PPA, runtime, timing slack, area, and power) from actual tool runs.

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Set project root
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_ROOT / "artifacts" / "datasets" / "v1"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset dir: {DATASET_DIR}")

## 1. Load Versioned Datasets (Raw & Normalized)

In [ ]:
raw_path = DATASET_DIR / "eda_experiments_raw.parquet"
norm_path = DATASET_DIR / "eda_experiments_normalized.parquet"

df_raw = pd.read_parquet(raw_path)
df_norm = pd.read_parquet(norm_path)

print(f"Raw dataset: {df_raw.shape[0]} samples, {df_raw.shape[1]} features")
print(f"Normalized dataset: {df_norm.shape[0]} samples, {df_norm.shape[1]} features")
df_raw.head()

## 2. Benchmark Design Breakdown & Success Rates

In [ ]:
design_summary = df_raw.groupby("design_name").agg(
    total_runs=("run_id", "count"),
    completed=("is_success", "sum"),
    mean_cells=("netlist_total_cells", "mean"),
    mean_synth_runtime=("target_synthesis_runtime_s", "mean"),
    mean_pd_runtime=("target_placement_runtime_s", "mean")
).reset_index()

design_summary["success_rate_pct"] = (design_summary["completed"] / design_summary["total_runs"]) * 100
design_summary

## 3. Correlation with Target Outcomes (Runtime, Area, Slack)

In [ ]:
targets = [
    "target_synthesis_runtime_s",
    "target_placement_runtime_s",
    "target_total_runtime_s",
    "target_cell_area_um2",
    "target_wns_ns"
]

numeric_df = df_raw.select_dtypes(include=[np.number])
features = [c for c in numeric_df.columns if not c.startswith("target_") and not c.startswith("is_")]

corr_matrix = numeric_df.corr()
top_synth_corrs = corr_matrix.loc[features, "target_synthesis_runtime_s"].abs().sort_values(ascending=False).head(10)
print("Top 10 features correlated with Synthesis Runtime (|r|):")
print(top_synth_corrs)

top_area_corrs = corr_matrix.loc[features, "target_cell_area_um2"].abs().sort_values(ascending=False).head(10)
print("\nTop 10 features correlated with Cell Area (|r|):")
print(top_area_corrs)

## 4. Multicollinearity & Redundancy Analysis (|r| >= 0.85)

In [ ]:
high_corr_path = DATASET_DIR / "highly_correlated_features.json"
with open(high_corr_path, "r") as f:
    high_corr = json.load(f)

print(f"Total redundant/collinear feature pairs (|r| >= 0.85): {len(high_corr)}")
pd.DataFrame(high_corr[:15])

## 5. Grouped Dataset Split Verification (Zero Leakage Check)

In [ ]:
split_summary_path = DATASET_DIR / "splits" / "split_summary.json"
with open(split_summary_path, "r") as f:
    split_summary = json.load(f)

print("Train Designs:     ", split_summary["train_designs"])
print("Validation Designs:", split_summary["val_designs"])
print("Test Designs:      ", split_summary["test_designs"])

train_set = set(split_summary["train_designs"])
val_set = set(split_summary["val_designs"])
test_set = set(split_summary["test_designs"])

# Rigorous Leakage Assertions
assert len(train_set.intersection(val_set)) == 0, "LEAKAGE DETECTED: Train & Val!"
assert len(train_set.intersection(test_set)) == 0, "LEAKAGE DETECTED: Train & Test!"
assert len(val_set.intersection(test_set)) == 0, "LEAKAGE DETECTED: Val & Test!"
print("\nSUCCESS: Zero design leakage between training, validation, and test partitions!")